In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print(" Ready")

 Ready


In [2]:
!pip install -q ddgs
print("✅ ddgs installed")

✅ ddgs installed



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%%writefile tools/llm_clients.py
import os
import time
import threading
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from tavily import TavilyClient
from ddgs import DDGS

load_dotenv()

GROQ_KEY = os.getenv("GROQ_API_KEY")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")
DEMO_MODE = os.getenv("DEMO_MODE", "false").lower() == "true"

assert GROQ_KEY, "Missing GROQ_API_KEY"
assert GEMINI_KEY, "Missing GEMINI_API_KEY"
assert TAVILY_KEY, "Missing TAVILY_API_KEY"

reasoning_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
gemini_client = genai.Client(api_key=GEMINI_KEY)
tavily_client = TavilyClient(api_key=TAVILY_KEY)

# Concurrency cap (Day 19, per mentor feedback): regardless of how many
# sub-questions the Supervisor dispatches via Send, only this many search
# calls actually run at once. DEMO_MODE forces fully sequential (1) for
# maximum reliability during a live demo; otherwise capped at 2 as advised.
MAX_CONCURRENT_SEARCHES = 1 if DEMO_MODE else 2
_search_semaphore = threading.Semaphore(MAX_CONCURRENT_SEARCHES)


def is_rate_limit_error(e: Exception) -> bool:
    msg = str(e).lower()
    return "429" in msg or "rate limit" in msg or "resource_exhausted" in msg


def call_llm(prompt: str, max_retries: int = 2) -> str:
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return reasoning_llm.invoke(prompt).content
        except Exception as e:
            last_error = e
            if is_rate_limit_error(e) and attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            raise
    raise last_error


def _search_duckduckgo(query: str) -> dict:
    """Third fallback tier. Free, keyless, but unofficial (scrapes DuckDuckGo's
    HTML) — more prone to blocking than a real API, so it's a last resort,
    not a primary choice."""
    results = list(DDGS().text(query, max_results=3))
    sources = [
        {"title": r.get("title", ""), "url": r.get("href") or r.get("url", "")}
        for r in results
    ]
    answer = " ".join(r.get("body", "")[:300] for r in results)
    return {"answer": answer, "sources": sources, "provider": "duckduckgo"}


def research_with_fallback(query: str) -> dict:
    """Gemini -> Tavily -> DuckDuckGo. Concurrency-limited via a semaphore so
    no more than MAX_CONCURRENT_SEARCHES calls hit any provider at once,
    regardless of how many parallel graph branches call this simultaneously."""
    with _search_semaphore:
        try:
            resp = gemini_client.models.generate_content(
                model="gemini-3.6-flash",
                contents=query,
                config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
            )
            sources = []
            if resp.candidates[0].grounding_metadata:
                sources = [
                    {"title": c.web.title, "url": c.web.uri}
                    for c in resp.candidates[0].grounding_metadata.grounding_chunks
                ]
            return {"answer": resp.text, "sources": sources, "provider": "gemini"}
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            print(f"⚠️ Gemini unavailable ({type(e).__name__}), falling back to Tavily...")

        try:
            results = tavily_client.search(query=query, max_results=3)
            sources = [{"title": r["title"], "url": r["url"]} for r in results["results"]]
            answer = " ".join(r.get("content", "")[:300] for r in results["results"])
            return {"answer": answer, "sources": sources, "provider": "tavily"}
        except Exception as e:
            print(f" Tavily unavailable ({type(e).__name__}), falling back to DuckDuckGo...")

        try:
            return _search_duckduckgo(query)
        except Exception as e:
            print(f"DuckDuckGo also unavailable ({type(e).__name__})")
            return {"answer": "", "sources": [], "provider": "none"}

Overwriting tools/llm_clients.py


In [4]:
%%writefile tests/test_llm_clients.py
import tools.llm_clients as llc


class FakeResponse:
    content = "ok"


def test_is_rate_limit_error_detects_variants():
    assert llc.is_rate_limit_error(Exception("429 Too Many Requests"))
    assert llc.is_rate_limit_error(Exception("RESOURCE_EXHAUSTED: quota"))
    assert llc.is_rate_limit_error(Exception("rate limit reached"))
    assert not llc.is_rate_limit_error(Exception("connection reset"))
    assert not llc.is_rate_limit_error(Exception("invalid API key"))


def test_call_llm_retries_on_rate_limit_then_succeeds(monkeypatch):
    calls = {"count": 0}
    class FlakyLLM:
        def invoke(self, prompt):
            calls["count"] += 1
            if calls["count"] < 2:
                raise Exception("429 rate limit")
            return FakeResponse()
    monkeypatch.setattr(llc, "reasoning_llm", FlakyLLM())
    monkeypatch.setattr(llc.time, "sleep", lambda s: None)
    result = llc.call_llm("prompt", max_retries=2)
    assert result == "ok"
    assert calls["count"] == 2


def test_call_llm_does_not_retry_non_rate_limit_errors(monkeypatch):
    calls = {"count": 0}
    class BrokenLLM:
        def invoke(self, prompt):
            calls["count"] += 1
            raise ValueError("unrelated failure")
    monkeypatch.setattr(llc, "reasoning_llm", BrokenLLM())
    try:
        llc.call_llm("prompt", max_retries=2)
        assert False
    except ValueError:
        pass
    assert calls["count"] == 1


def test_call_llm_exhausts_retries_and_raises(monkeypatch):
    calls = {"count": 0}
    class AlwaysLimitedLLM:
        def invoke(self, prompt):
            calls["count"] += 1
            raise Exception("429 rate limit")
    monkeypatch.setattr(llc, "reasoning_llm", AlwaysLimitedLLM())
    monkeypatch.setattr(llc.time, "sleep", lambda s: None)
    try:
        llc.call_llm("prompt", max_retries=2)
        assert False
    except Exception as e:
        assert "429" in str(e)
    assert calls["count"] == 3


def test_research_falls_through_to_duckduckgo(monkeypatch):
    class FailingGemini:
        class models:
            @staticmethod
            def generate_content(**kwargs):
                raise llc.genai_errors.ClientError(500, {"error": {"message": "down"}}, None)
    class FailingTavily:
        def search(self, **kwargs):
            raise Exception("down")
    monkeypatch.setattr(llc, "gemini_client", FailingGemini())
    monkeypatch.setattr(llc, "tavily_client", FailingTavily())
    monkeypatch.setattr(llc, "_search_duckduckgo", lambda q: {"answer": "ddg result", "sources": [], "provider": "duckduckgo"})

    result = llc.research_with_fallback("test query")
    assert result["provider"] == "duckduckgo"


def test_all_three_providers_fail_returns_empty_not_crash(monkeypatch):
    class FailingGemini:
        class models:
            @staticmethod
            def generate_content(**kwargs):
                raise llc.genai_errors.ClientError(500, {"error": {"message": "down"}}, None)
    class FailingTavily:
        def search(self, **kwargs):
            raise Exception("down")
    monkeypatch.setattr(llc, "gemini_client", FailingGemini())
    monkeypatch.setattr(llc, "tavily_client", FailingTavily())
    monkeypatch.setattr(llc, "_search_duckduckgo", lambda q: (_ for _ in ()).throw(Exception("down")))

    result = llc.research_with_fallback("test query")
    assert result["provider"] == "none"
    assert result["answer"] == ""

Overwriting tests/test_llm_clients.py


In [5]:
%%writefile graph/build_graph.py
from langgraph.graph import StateGraph, START, END
from state.schema import ResearchState
from agents.supervisor_agent import supervisor_node, route_from_supervisor
from agents.web_search_agent import search_single_node
from agents.data_extraction_agent import data_extraction_node
from agents.fact_checking_agent import fact_checking_node
from agents.summarization_agent import summarization_node
from agents.report_generator_agent import report_generator_node


def build_graph(checkpointer=None, interrupt_before=None):
    """checkpointer + interrupt_before are optional (Day 19) — when provided,
    the graph pauses before the named node(s) so a human can review state
    before execution continues. Omitted, the graph runs fully autonomously
    as it has since Day 13."""
    graph = StateGraph(ResearchState)

    graph.add_node("supervisor", supervisor_node)
    graph.add_node("web_search_single", search_single_node)
    graph.add_node("data_extraction", data_extraction_node)
    graph.add_node("fact_checking", fact_checking_node)
    graph.add_node("summarization", summarization_node)
    graph.add_node("report_generator", report_generator_node)

    graph.add_edge(START, "supervisor")
    graph.add_conditional_edges("supervisor", route_from_supervisor, {
        "web_search_single": "web_search_single",
        "data_extraction": "data_extraction",
        "fact_checking": "fact_checking",
        "summarization": "summarization",
        "report_generator": "report_generator",
        "done": END,
        "failed": END,
    })
    for node in ["web_search_single", "data_extraction", "fact_checking", "summarization", "report_generator"]:
        graph.add_edge(node, "supervisor")

    compile_kwargs = {}
    if checkpointer is not None:
        compile_kwargs["checkpointer"] = checkpointer
    if interrupt_before is not None:
        compile_kwargs["interrupt_before"] = interrupt_before

    return graph.compile(**compile_kwargs)

Overwriting graph/build_graph.py


In [6]:
from langgraph.checkpoint.memory import MemorySaver
from graph.build_graph import build_graph

checkpointer = MemorySaver()  
app_review = build_graph(checkpointer=checkpointer, interrupt_before=["report_generator"])

config = {"configurable": {"thread_id": "review-session-1"}}

initial_state = {
    "query": "What are the main safety risks of solid-state EV batteries?",
    "plan": [], "next_agent": "", "iteration_count": 0, "needs_more_research": False,
    "search_results": [], "extracted_facts": [], "fact_check_results": [], "summaries": [],
    "errors": [], "final_report": None, "model_usage_log": [],
}

result = app_review.invoke(initial_state, config=config)

state_now = app_review.get_state(config)
print("Paused before:", state_now.next)
print(f"\n{len(state_now.values['summaries'])} sub-topic summaries ready for review:\n")
for s in state_now.values["summaries"]:
    print(f"— {s['sub_question']}")
    print(f"  {s['summary'][:200]}...\n")

2026-10-27 19:31:25,140 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=4
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


⚠️ Gemini unavailable (ClientError), falling back to Tavily...
⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-10-27 19:31:29,554 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-27 19:31:30,015 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0


⚠️ Gemini unavailable (ClientError), falling back to Tavily...
⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-10-27 19:31:32,165 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-27 19:31:32,724 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-27 19:31:32,728 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-10-27 19:31:33,292 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How do solid electrolyte interfaces degrade under thermal stress and what are the resulting failure modes? facts_found=10
2026-10-27 19:31:33,803 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the specific mechanical risks associated with dendrite formation in solid-state batteries during high-rate charging? facts_found=8
2026-10-27 19:31:34,111 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How does the thermal runaway behavior of solid-state ba

Paused before: ('report_generator',)

4 sub-topic summaries ready for review:

— How do solid electrolyte interfaces degrade under thermal stress and what are the resulting failure modes?
  It is confirmed that temperature strongly influences reaction kinetics, creep behavior, and the mechanical stress state at the lithium-solid electrolyte interface, with cooling from a stress-free high...

— What are the specific mechanical risks associated with dendrite formation in solid-state batteries during high-rate charging?
  Dendrites, which are filaments of lithium metal, form at the anode interface with the solid electrolyte during high-rate charging. These filaments can propagate through the solid electrolyte towards t...

— How does the thermal runaway behavior of solid-state batteries differ from that of conventional liquid-electrolyte lithium-ion batteries?
  Conventional lithium-ion batteries utilize flammable liquid electrolytes based on organic solvents, whereas solid-state batterie

In [7]:
print("Reviewing summaries above. Approving continuation to report generation...\n")

final_result = app_review.invoke(None, config=config) 
print("Report generated. Length:", len(final_result["final_report"]), "chars")
print("\n" + final_result["final_report"][:500] + "...")

Reviewing summaries above. Approving continuation to report generation...



2026-10-27 19:33:21,941 | INFO | research_system | [report_generator_agent] report_done method=llm sub_topics=4 total_claims=31
2026-10-27 19:33:21,947 | INFO | research_system | [supervisor_agent] pipeline_complete iteration=6


Report generated. Length: 6781 chars

# Safety Risks of Solid-State EV Batteries

Solid-state batteries represent a significant evolution in electric vehicle technology, promising improved energy density and potentially enhanced safety profiles compared to conventional lithium-ion cells. However, the transition from liquid to solid electrolytes introduces distinct mechanical and thermal challenges. This report examines the primary safety risks associated with solid-state EV batteries, focusing on interfacial degradation, dendrite fo...


In [ ]:
%%writefile tests/test_human_in_the_loop.py
from langgraph.checkpoint.memory import MemorySaver
import agents.supervisor_agent as sup


def test_graph_pauses_before_report_generator(monkeypatch):
    import graph.build_graph as gb

    monkeypatch.setattr(sup, "generate_plan", lambda q: ["sub-q 1"])
    monkeypatch.setattr(gb, "search_single_node", lambda state: {
        "search_results": [{"sub_question": state["query"], "status": "ok"}],
        "model_usage_log": [{"agent": "web_search_agent"}]})
    monkeypatch.setattr(gb, "data_extraction_node", lambda state: {
        "extracted_facts": [{"claim": "x"}], "model_usage_log": [{"agent": "data_extraction_agent"}]})
    monkeypatch.setattr(gb, "fact_checking_node", lambda state: {
        "fact_check_results": [{"claim": "x", "status": "CONFIRMED"}],
        "model_usage_log": [{"agent": "fact_checking_agent"}]})
    monkeypatch.setattr(gb, "summarization_node", lambda state: {
        "summaries": [{"sub_question": "q1", "summary": "a summary"}],
        "model_usage_log": [{"agent": "summarization_agent"}]})
    monkeypatch.setattr(gb, "report_generator_node", lambda state: {
        "final_report": "the final report", "model_usage_log": [{"agent": "report_generator_agent"}]})

    checkpointer = MemorySaver()
    app = gb.build_graph(checkpointer=checkpointer, interrupt_before=["report_generator"])
    config = {"configurable": {"thread_id": "test-thread-1"}}

    initial_state = {
        "query": "test query", "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }
    result = app.invoke(initial_state, config=config)

    state = app.get_state(config)
    assert state.next == ("report_generator",)
    assert result.get("final_report") is None  # confirms it paused BEFORE report generation
    assert len(state.values["summaries"]) == 1  # but summaries ARE ready to review

    final = app.invoke(None, config=config)
    assert final["final_report"] == "the final report"